# 03 — Task 2.1.2: Pre-trained Transformer Baseline
Applies `distilbert-base-uncased-finetuned-sst-2-english` to the IMDB test set **without any fine-tuning**.

In [ ]:
import sys
sys.path.insert(0, '..')

from tqdm import tqdm
import pandas as pd

from src.utils import load_data, evaluate_predictions, save_results

test_texts, test_labels = load_data('test')
print(f'Test set: {len(test_texts):,} reviews')

## 1. Load pipeline

In [ ]:
from transformers import pipeline

MODEL = 'distilbert-base-uncased-finetuned-sst-2-english'

# truncation=True handles reviews longer than 512 tokens
classifier = pipeline(
    'sentiment-analysis',
    model=MODEL,
    truncation=True,
    max_length=512,
    device=-1,  # CPU; set to 0 for GPU
)
print(f'Loaded: {MODEL}')

## 2. Run inference

In [ ]:
BATCH_SIZE = 32

raw_preds = []
for i in tqdm(range(0, len(test_texts), BATCH_SIZE), desc='DistilBERT inference'):
    batch = test_texts[i:i+BATCH_SIZE]
    results = classifier(batch)
    raw_preds.extend(results)

# Map POSITIVE/NEGATIVE → pos/neg
preds = ['pos' if r['label'] == 'POSITIVE' else 'neg' for r in raw_preds]
print(f'Done. Sample: {raw_preds[:3]}')

## 3. Evaluate

In [ ]:
metrics = evaluate_predictions(test_labels, preds)
print('Pre-trained DistilBERT (SST-2):', metrics)

save_results(
    task='2.1.2',
    approach='DistilBERT (pre-trained, SST-2)',
    metrics=metrics,
    preprocessing='truncation to 512 tokens',
    notes='No fine-tuning; zero-shot transfer from SST-2 to IMDB'
)

## 4. Confidence score distribution

In [ ]:
import matplotlib.pyplot as plt

scores = [r['score'] for r in raw_preds]
correct = [p == t for p, t in zip(preds, test_labels)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([s for s, c in zip(scores, correct) if c],
        bins=40, alpha=0.6, label='Correct', color='#4878D0')
ax.hist([s for s, c in zip(scores, correct) if not c],
        bins=40, alpha=0.6, label='Incorrect', color='#EE854A')
ax.set_xlabel('Confidence score')
ax.set_ylabel('Count')
ax.set_title('DistilBERT — Confidence distribution (correct vs incorrect)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/fig_distilbert_baseline_confidence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Accuracy: {metrics['accuracy']} | F1: {metrics['f1']}")